<style>
.markdown-content {
    direction: rtl;
    text-align: right;
}
.markdown-content pre {
    direction: ltr;
    text-align: left;
}
.markdown-content code {
    direction: ltr;
    text-align: left;
}
</style>

<div class="markdown-content">

# Class Imbalance

**Class Imbalance** زمانی اتفاق می‌افتد که تعداد نمونه‌های کلاس‌های یک Dataset به‌شدت متفاوت باشد.

### مثال

در یک Dataset شامل ۱۰٬۰۰۰ تراکنش:

- Normal: ۹٬۹۰۰ نمونه
- Fraud: ۱۰۰ نمونه

### مشکل اصلی

ممکن است مدل با پیش‌بینی همیشگی کلاس اکثریت، Accuracy بالایی بگیرد؛ بدون اینکه کلاس اقلیت را به‌درستی شناسایی کند.

### روش‌های مدیریت

| روش | عملکرد |
|---|---|
| `Random Oversampling` | افزایش نمونه‌های کلاس اقلیت با نمونه‌برداری مجدد |
| `Random Undersampling` | کاهش نمونه‌های کلاس اکثریت |
| `SMOTE` | تولید نمونه‌های مصنوعی برای کلاس اقلیت |
| `class_weight` | افزایش وزن کلاس اقلیت هنگام آموزش |

### معیارهای ارزیابی مهم

- `Precision`
- `Recall`
- `F1-score`
- `Confusion Matrix`
- `PR-AUC`

### نکات مهم

1. Accuracy به‌تنهایی برای Dataset نامتوازن کافی نیست.
2. Resampling باید فقط روی Train انجام شود.
3. Test Dataset باید توزیع واقعی داده را حفظ کند.
4. متعادل کردن تعداد کلاس‌ها همیشه باعث بهتر شدن مدل نمی‌شود.
5. انتخاب روش مناسب به مسئله و هزینه خطاها بستگی دارد.

</div>

<style>
.markdown-content {
    direction: rtl;
    text-align: right;
}
.markdown-content pre {
    direction: ltr;
    text-align: left;
}
.markdown-content code {
    direction: ltr;
    text-align: left;
}
</style>

<div class="markdown-content">

# تمرین موضوع ۲۳ — Class Imbalance

با استفاده از `make_classification` یک Dataset نامتوازن بساز.

### مراحل تمرین

**۱. ساخت Dataset**

از پارامترهای زیر استفاده کن:

```python
X, y = make_classification(
    n_samples=5000,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    weights=[0.95, 0.05],
    random_state=42
)
```

**۲. بررسی توزیع کلاس‌ها**

با استفاده از `Counter` تعداد نمونه‌های هر کلاس را نمایش بده.

**۳. تقسیم داده‌ها**

از `train_test_split` استفاده کن:

- `test_size=0.2`
- `random_state=42`
- `stratify=y`

**۴. ساخت سه مدل**

- مدل اول: `LogisticRegression` معمولی
- مدل دوم: `LogisticRegression(class_weight="balanced")`
- مدل سوم: `LogisticRegression` همراه با `SMOTE`

برای مدل سوم از `imblearn.pipeline.Pipeline` استفاده کن تا SMOTE فقط روی داده‌های Train در زمان آموزش اعمال شود.

**۵. ارزیابی مدل‌ها**

برای هر سه مدل موارد زیر را محاسبه کن:

- Accuracy
- Precision
- Recall
- F1-score
- Confusion Matrix

**۶. مقایسه**

نتایج را در یک DataFrame نمایش بده و عملکرد سه مدل را مقایسه کن.

</div>

In [1]:
import pandas as pd

from collections import Counter
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline


# 2. Generate Imbalanced Dataset

X, y = make_classification(
    n_samples=5000,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    weights=[0.95, 0.05],
    random_state=42,
)


# 3. Check Class Distribution

print("Original Class Distribution:")
print(Counter(y))


# 4. Split Dataset

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


print("\nTraining Class Distribution:")
print(Counter(y_train))

print("\nTesting Class Distribution:")
print(Counter(y_test))


# 5. Define Models

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Balanced Logistic Regression": LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=42
    ),
    "Logistic Regression + SMOTE": Pipeline(
        [
            ("smote", SMOTE(random_state=42)),
            ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
        ]
    ),
}


# 6. Train and Evaluate Models

results = []

for name, model in models.items():
    # Train model
    model.fit(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_test)

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(y_test, y_pred, zero_division=0)

    recall = recall_score(y_test, y_pred, zero_division=0)

    f1 = f1_score(y_test, y_pred, zero_division=0)

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)

    # Store results
    results.append(
        {
            "Model": name,
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "F1-score": f1,
        }
    )

    print(f"\n{name}")
    print("Confusion Matrix:")
    print(cm)


# 7. Compare Results

results_df = pd.DataFrame(results)

print("\nModel Comparison:")
print(results_df.to_string(index=False))


# 8. Find Best Model Based on F1-score

best_model = results_df.loc[results_df["F1-score"].idxmax()]

print("\nBest Model:")
print(best_model)

Original Class Distribution:
Counter({np.int64(0): 4725, np.int64(1): 275})

Training Class Distribution:
Counter({np.int64(0): 3780, np.int64(1): 220})

Testing Class Distribution:
Counter({np.int64(0): 945, np.int64(1): 55})

Logistic Regression
Confusion Matrix:
[[942   3]
 [ 51   4]]

Balanced Logistic Regression
Confusion Matrix:
[[696 249]
 [  8  47]]

Logistic Regression + SMOTE
Confusion Matrix:
[[704 241]
 [  8  47]]

Model Comparison:
                       Model  Accuracy  Precision   Recall  F1-score
         Logistic Regression     0.946   0.571429 0.072727  0.129032
Balanced Logistic Regression     0.743   0.158784 0.854545  0.267806
 Logistic Regression + SMOTE     0.751   0.163194 0.854545  0.274052

Best Model:
Model        Logistic Regression + SMOTE
Accuracy                           0.751
Precision                       0.163194
Recall                          0.854545
F1-score                        0.274052
Name: 2, dtype: object


In [ ]:
# 9. Analysis

# A. Accuracy alone is not sufficient for evaluating
# imbalanced classification datasets.

# B. class_weight="balanced" increases the importance
# of minority-class samples during model training.

# C. SMOTE generates synthetic minority-class samples
# using neighboring samples.

# D. Recall measures how many actual positive samples
# were correctly identified.

# E. Precision measures how many predicted positive samples
# were actually positive.

# F. F1-score balances Precision and Recall.

# G. SMOTE is applied only to training data through
# imblearn Pipeline. Test data remains unchanged.

# H. The model with the highest F1-score performed best
# according to this experiment's F1-score criterion.

# I. Higher Recall does not always mean a better model.
# Precision and the cost of false positives should also
# be considered.